Scarica sia i dati di raccolta urbana rifiuti (comune per comune) sia i dati
sugli imballaggi (acciaio, alluminio, carta, legno, bioplastica, plastica,
vetro) da osservatorioraccoltadifferenziata.it, per gli anni 2020-2023.

Salva:
  - CSV per ogni combinazione anno/regione della raccolta urbana (downloaded_csv/)
  - CSV finale raccolta urbana (raccolta_urbana_totale_2020_2023.csv)
  - CSV per ogni combinazione materiale/anno degli imballaggi (downloaded_csv_imballaggi/)
  - CSV finale imballaggi (imballaggi_totale_2020_2023.csv)


In [2]:
import os
import time
import json
import requests
import pandas as pd

In [4]:

# ----------------------------------------------------------------------
# CONFIGURAZIONE CONDIVISA
# ----------------------------------------------------------------------

BASE = "https://osservatorioraccoltadifferenziata.it/wp-content/plugins/osservatorio/ajax"
REFERER_URBANA = "https://osservatorioraccoltadifferenziata.it/dati-raccolta-urbana/"

ANNI = ["2020", "2021", "2022", "2023"]

REGIONI = {
    "Abruzzo": "13", 
    "Basilicata": "17", 
    "Calabria": "18", 
    "Campania": "15",
    "Emilia-Romagna": "08", 
    "Friuli-Venezia Giulia": "06", 
    "Lazio": "12",
    "Liguria": "07", 
    "Lombardia": "03", 
    "Marche": "11", 
    "Molise": "14",
    "Piemonte": "01", 
    "Puglia": "16", 
    "Sardegna": "20", 
    "Sicilia": "19",
    "Toscana": "09", 
    "Trentino-Alto Adige": "04", 
    "Umbria": "10",
    "Valle d'Aosta": "02", 
    "Veneto": "05",
}

MATERIALI = {
    "Acciaio":     ("ricrea",    "https://osservatorioraccoltadifferenziata.it/dati-imballaggi/consorzio-ricrea/"),
    "Alluminio":   ("cial",      "https://osservatorioraccoltadifferenziata.it/dati-imballaggi/consorzio-cial/"),
    "Carta":       ("comieco",   "https://osservatorioraccoltadifferenziata.it/dati-imballaggi/consorzio-comieco/"),
    "Legno":       ("rilegno",   "https://osservatorioraccoltadifferenziata.it/dati-imballaggi/consorzio-rilegno/"),
    "Bioplastica": ("biorepack", "https://osservatorioraccoltadifferenziata.it/dati-imballaggi/consorzio-biorepack/"),
    "Plastica":    ("corepla",   "https://osservatorioraccoltadifferenziata.it/dati-imballaggi/consorzio-corepla/"),
    "Vetro":       ("coreve",    "https://osservatorioraccoltadifferenziata.it/dati-imballaggi/consorzio-coreve/"),
}

OUTPUT_DIR_URBANA = "downloaded_csv"
FINAL_FILE_URBANA = "raccolta_urbana_totale_2020_2023.csv"

OUTPUT_DIR_IMBALLAGGI = "downloaded_csv_imballaggi"
FINAL_FILE_IMBALLAGGI = "imballaggi_totale_2020_2023.csv"

TIMEOUT_SECONDI = 30
PAUSA_TRA_RICHIESTE = 1.5

HEADERS_BASE = {
    "Accept": "*/*",
    "Accept-Language": "it,it-IT;q=0.9,en-US;q=0.8,en;q=0.7",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://osservatorioraccoltadifferenziata.it",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

In [5]:

os.makedirs(OUTPUT_DIR_URBANA, exist_ok=True)
os.makedirs(OUTPUT_DIR_IMBALLAGGI, exist_ok=True)

In [6]:
"""Funzione per creare una sessione con un dato Referer.
    Usata sia per i dati della raccolta urbana sia per i dati degli imballaggi."""

def crea_sessione(referer):
    session = requests.Session()
    session.headers.update(HEADERS_BASE)
    session.headers["Referer"] = referer
    session.get(referer, timeout=TIMEOUT_SECONDI)
    return session


In [7]:
"""interrogazione di ajax_getProvinces.php (api del sito) per ogni regione e costruisce un
    dizionario {codice_provincia_3_cifre: nome_regione}. 
    Usata solo per gli imballaggi (la raccolta urbana ha già la regione)."""

def costruisci_mappa_provincia_regione(session):
    mappa = {}
    print("Costruisco la mappa provincia -> regione...")
    for regione_nome, regione_codice in REGIONI.items():
        payload = {"region": regione_codice}
        r = session.post(f"{BASE}/ajax_getProvinces.php", data=json.dumps(payload), timeout=TIMEOUT_SECONDI)
        r.raise_for_status()
        data = r.json()
        for prov in data.get("provinces", []):
            codice_prov = prov.get("istat_provincia", "")
            if codice_prov:
                mappa[codice_prov] = regione_nome
        time.sleep(0.5)
    print(f"Mappa costruita: {len(mappa)} province -> regione.")
    return mappa



In [ ]:

# ----------------------------------------------------------------------
# PARTE 1: RACCOLTA URBANA
# ----------------------------------------------------------------------

def scarica_dati_regione_anno(session, anno, regione_nome, regione_codice):
    """Chiama l'endpoint dati e restituisce un DataFrame con i dati del comune."""
    payload = {
        "year": anno,
        "region": regione_codice,
        "province": "",
        "city": "",
    }
    r = session.post(
        f"{BASE}/ajax_dataUrbanCollectionGetResults.php",
        data=json.dumps(payload),
        timeout=TIMEOUT_SECONDI,
    )
    r.raise_for_status()
    data = r.json()

    if data.get("err") != "0":
        raise RuntimeError(f"L'API ha restituito un errore: {data}")

    fields = data.get("fields1", [])
    results = data.get("results1", [])

    if not results:
        df_vuoto = pd.DataFrame(columns=fields)
        df_vuoto["Regione"] = pd.Series(dtype=str)
        return df_vuoto

    righe = [{fields[int(k)]: v for k, v in rec.items()} for rec in results]
    df = pd.DataFrame(righe).astype(str)
    df["Regione"] = regione_nome
    return df


In [ ]:
def main_raccolta_urbana():
    session = crea_sessione(REFERER_URBANA)

    file_creati = []
    errori = []

    for anno in ANNI:
        for regione_nome, regione_codice in REGIONI.items():
            filename = f"dati_{regione_nome}_{anno}.csv".replace(" ", "_").replace("'", "")
            filepath = os.path.join(OUTPUT_DIR_URBANA, filename)

            if os.path.exists(filepath):
                print(f"Già presente, salto: {filename}")
                file_creati.append(filepath)
                continue

            print(f"[URBANA] Scarico: anno={anno}, regione={regione_nome}...")
            try:
                df = scarica_dati_regione_anno(session, anno, regione_nome, regione_codice)
                df.to_csv(filepath, index=False, sep=";", encoding="utf-8-sig")
                file_creati.append(filepath)
                print(f"  -> salvato {filename} ({len(df)} righe)")
            except Exception as e:
                print(f"  [ERRORE] anno={anno}, regione={regione_nome}: {e}")
                errori.append((anno, regione_nome, str(e)))

            time.sleep(PAUSA_TRA_RICHIESTE)

    print("\n[URBANA] Unione dei file in corso...")
    if file_creati:
        df_finale = pd.concat(
            [pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str) for f in file_creati],
            ignore_index=True,
        )
        df_finale.to_csv(FINAL_FILE_URBANA, index=False, sep=";", encoding="utf-8-sig")
        print(f"File finale salvato: {FINAL_FILE_URBANA} ({len(df_finale)} righe totali)")
    else:
        print("Nessun file scaricato, niente da unire.")

    print("\n=== RIEPILOGO RACCOLTA URBANA ===")
    print(f"File scaricati con successo: {len(file_creati)}")
    print(f"Errori: {len(errori)}")
    for anno, regione, err in errori:
        print(f"  - {anno} / {regione}: {err}")


Attenzione quello qui indicato è riportato all'interno del sito:

"Nella presente sezione è possibile consultare i dati relativi ai quantitativi di rifiuti urbani raccolti dai gestori dei servizi comunali per singolo CER (codice identificativo rifiuti), al Comune e alla popolazione interessata dalla raccolta.
I dati esposti sono stati rilevati da ANCI attraverso le seguenti fonti informative:

le Regioni e le Province Autonome, che hanno trasmesso i dati annuali dei rifiuti urbani raccolti, classificati per CER;
il Catasto Nazionale Rifiuti di ISPRA, utilizzato per coprire eventuali vuoti informativi dei dati trasmessi dalle Regioni.
ANCI ha provveduto successivamente ad elaborare ed uniformare i dati pervenuti, con alcune specifiche:

* per alcune tipologie di rifiuti (ingombranti a recupero, spazzamento a recupero, autocompostaggio), come da notazione ISPRA, sono stati utilizzati codici non previsti nel DM 26 maggio 2016 (rispettivamente 20030_ è il CER assegnato alla raccolta degli ingombranti a recupero, 200380 è il CER assegnato allo spazzamento a recupero, 200109 è il CER assegnato all’autocompostaggio);

* per la raccolta multimateriale sono considerate all’interno del codice 150106 tutte le singole voci merceologiche di composizione della raccolta degli imballaggi misti (carta, plastica, vetro, ecc.), compreso gli scarti, trattando di raccolta dei rifiuti e non di avvio a riciclo;

* i dati scaricati dal Catasto Nazionale Rifiuti ISPRA ed utilizzati per colmare eventuali vuoti dell’informativa regionale, organizzati per frazione merceologica, sono stati ricodificati per codice rifiuto, assegnando alle singole frazioni i seguenti codici rifiuti: Carta (codice 15200101), Plastica (codice 15200139), Metalli (codice 15200140), Legno (codice 15200138), Organico (codice 20010802), Vetro (codice 15200102).

Al fine di semplificare l’esposizione di tutti i dati di raccolta differenziata previsti nel DM 26 Maggio 2016, si è scelto di accorpare all’interno del codice convenzionalmente riconosciuto 777777 tutte le raccolte differenziate minoritarie costituite dai CER di seguito elencati: 80318, 150105, 150110, 150111, 160103, 160107, 160211, 160213, 160214, 160215, 160216, 160504, 160505, 170107, 170904, 200113, 200114, 200115, 200117, 200119, 200125, 200126, 200127, 200128, 200129, 200130, 200131, 200132, 200137, 200141, 200202, 200203, 200302, 200380, 20020102."

In [10]:
main_raccolta_urbana()

Già presente, salto: dati_Abruzzo_2020.csv
Già presente, salto: dati_Basilicata_2020.csv
Già presente, salto: dati_Calabria_2020.csv
Già presente, salto: dati_Campania_2020.csv
Già presente, salto: dati_Emilia-Romagna_2020.csv
Già presente, salto: dati_Friuli-Venezia_Giulia_2020.csv
Già presente, salto: dati_Lazio_2020.csv
Già presente, salto: dati_Liguria_2020.csv
Già presente, salto: dati_Lombardia_2020.csv
Già presente, salto: dati_Marche_2020.csv
Già presente, salto: dati_Molise_2020.csv
Già presente, salto: dati_Piemonte_2020.csv
Già presente, salto: dati_Puglia_2020.csv
Già presente, salto: dati_Sardegna_2020.csv
Già presente, salto: dati_Sicilia_2020.csv
Già presente, salto: dati_Toscana_2020.csv
Già presente, salto: dati_Trentino-Alto_Adige_2020.csv
Già presente, salto: dati_Umbria_2020.csv
Già presente, salto: dati_Valle_dAosta_2020.csv
Già presente, salto: dati_Veneto_2020.csv
Già presente, salto: dati_Abruzzo_2021.csv
Già presente, salto: dati_Basilicata_2021.csv
Già present

In [ ]:
# ----------------------------------------------------------------------
# PARTE 2: IMBALLAGGI
# ----------------------------------------------------------------------

def scarica_dati_materiale_anno(session, anno, materiale_nome, consorzio_slug, mappa_provincia_regione):
    payload = {
        "year": anno,
        "region": "",
        "province": "",
        "city": "",
        "consortium": consorzio_slug,
    }
    r = session.post(
        f"{BASE}/ajax_consortiumGetResults.php",
        data=json.dumps(payload),
        timeout=TIMEOUT_SECONDI,
    )
    r.raise_for_status()
    data = r.json()

    if data.get("err") != "0":
        raise RuntimeError(f"L'API ha restituito un errore: {data}")

    fields = data.get("fields", [])
    results = data.get("results", [])

    if not results:
        df_vuoto = pd.DataFrame(columns=fields)
        df_vuoto["Regione"] = pd.Series(dtype=str)
        df_vuoto["Materiale"] = pd.Series(dtype=str)
        df_vuoto["Anno"] = pd.Series(dtype=str)
        return df_vuoto

    righe = [{fields[int(k)]: v for k, v in rec.items()} for rec in results]
    df = pd.DataFrame(righe).astype(str)

    codice_provincia = df["Codice ISTAT"].astype(str).str.zfill(6).str[:3]
    df["Regione"] = codice_provincia.map(mappa_provincia_regione)

    df["Materiale"] = materiale_nome
    df["Anno"] = anno
    return df


In [12]:

def main_imballaggi():
    file_creati = []
    errori = []

    session_mappa = crea_sessione(list(MATERIALI.values())[0][1])
    mappa_provincia_regione = costruisci_mappa_provincia_regione(session_mappa)

    for materiale_nome, (consorzio_slug, referer) in MATERIALI.items():
        session = crea_sessione(referer)

        for anno in ANNI:
            filename = f"dati_{materiale_nome}_{anno}.csv"
            filepath = os.path.join(OUTPUT_DIR_IMBALLAGGI, filename)

            if os.path.exists(filepath):
                print(f"Già presente, salto: {filename}")
                file_creati.append(filepath)
                continue

            print(f"[IMBALLAGGI] Scarico: materiale={materiale_nome} ({consorzio_slug}), anno={anno}...")
            try:
                df = scarica_dati_materiale_anno(session, anno, materiale_nome, consorzio_slug, mappa_provincia_regione)
                df.to_csv(filepath, index=False, sep=";", encoding="utf-8-sig")
                file_creati.append(filepath)
                print(f"  -> salvato {filename} ({len(df)} righe)")
            except Exception as e:
                print(f"  [ERRORE] materiale={materiale_nome}, anno={anno}: {e}")
                errori.append((materiale_nome, anno, str(e)))

            time.sleep(PAUSA_TRA_RICHIESTE)

    print("\n[IMBALLAGGI] Unione dei file in corso...")
    if file_creati:
        df_finale = pd.concat(
            [pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str) for f in file_creati],
            ignore_index=True,
        )
        df_finale.to_csv(FINAL_FILE_IMBALLAGGI, index=False, sep=";", encoding="utf-8-sig")
        print(f"File finale salvato: {FINAL_FILE_IMBALLAGGI} ({len(df_finale)} righe totali)")
    else:
        print("Nessun file scaricato, niente da unire.")

    print("\n=== RIEPILOGO IMBALLAGGI ===")
    print(f"File scaricati con successo: {len(file_creati)}")
    print(f"Errori: {len(errori)}")
    for materiale, anno, err in errori:
        print(f"  - {materiale} / {anno}: {err}")



i consorzi presenti nel sito sono:

- RICREA (consorzio nazionale riciclo e recupero imballaggi acciaio)
- CiAl (consorzio imballaggi alluminio)
- Comieco (consorzio nazionale Recupero e riciclo degli imballaggi a base cellulosica) 
- rilegno
- biorepack
- corepla (consorzio nazionale per la raccolta, il riciclo e li recupero dei rifiuti di imballagio in plastica)
- CoReVE

Attenzione quello qui indicato è riportato all'interno del sito:

"Nella presente sezione vengono riportati i quantitativi conferiti ai Consorzi di filiera, nonché i corrispettivi economici erogati dai Consorzi a favore dei soggetti convenzionati. Tutti i dati riportati in questa sezione sono forniti dai Consorzi di filiera del CONAI, riferiti ad ogni bacino di conferimento, elaborati da ANCI e pubblicati nelle specifiche sezioni riferite a ciascun Consorzio. Conferimenti e corrispettivi, a livello comunale, sono stati stimati dividendo i ricavi e le quantità di ciascun bacino di appartenenza – associato ad ogni Soggetto Convenzionato per singolo Consorzio di filiera del CONAI – per la popolazione associata al bacino e moltiplicando il risultato medio per la popolazione del Comune di riferimento.
I dati esposti nella presente sezione, pertanto, potranno essere:

* puntualmente riferiti ai Comuni, nel caso in cui ad un bacino di conferimento è associato un solo Comune;
* stimati in base alla popolazione residente sull’intero bacino, nel caso in cui ad un bacino di conferimento sono associati più Comuni.
"

In [13]:
main_imballaggi()

Costruisco la mappa provincia -> regione...
Mappa costruita: 111 province -> regione.
Già presente, salto: dati_Acciaio_2020.csv
Già presente, salto: dati_Acciaio_2021.csv
Già presente, salto: dati_Acciaio_2022.csv
Già presente, salto: dati_Acciaio_2023.csv
Già presente, salto: dati_Alluminio_2020.csv
Già presente, salto: dati_Alluminio_2021.csv
Già presente, salto: dati_Alluminio_2022.csv
Già presente, salto: dati_Alluminio_2023.csv
Già presente, salto: dati_Carta_2020.csv
Già presente, salto: dati_Carta_2021.csv
Già presente, salto: dati_Carta_2022.csv
Già presente, salto: dati_Carta_2023.csv
Già presente, salto: dati_Legno_2020.csv
Già presente, salto: dati_Legno_2021.csv
Già presente, salto: dati_Legno_2022.csv
Già presente, salto: dati_Legno_2023.csv
Già presente, salto: dati_Bioplastica_2020.csv
Già presente, salto: dati_Bioplastica_2021.csv
Già presente, salto: dati_Bioplastica_2022.csv
Già presente, salto: dati_Bioplastica_2023.csv
Già presente, salto: dati_Plastica_2020.csv
Gi

In [ ]:
""" questo codice cancella le cartelle e i file creati """

"""
import shutil
shutil.rmtree("downloaded_csv_imballaggi", ignore_errors=True)
shutil.rmtree("downloaded_csv", ignore_errors=True)
shutil.rmtree("imballaggi_totale_2020_2023", ignore_errors=True)
shutil.rmtree("raccolta_urbana_totale_2020_2023", ignore_errors=True)
"""